In [1]:
import os

In [2]:
%pwd

'/Users/harjanag/Documents/End-To-End-ML-Project-with-MLflow/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/harjanag/Documents/End-To-End-ML-Project-with-MLflow'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from CATProject.constants import *
from CATProject.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config


In [75]:
import os
import ssl
import certifi
import urllib.request as request
import requests
import zipfile
from CATProject import logger
from CATProject.utils.common import get_size


In [70]:
# class DataIngestion:
#     def __init__(self, config:DataIngestionConfig):
#         self.config = config

#     def download_file(self):
#         if not os.path.exists(self.config.local_data_file):
#             # ctx = ssl.create_default_context(cafile=certifi.where())
#             filename, headers = request.urlretrieve(
#                 self.config.source_URL,
#                 self.config.local_data_file,
#                 # context = ctx
#             )
#             data = requests.get(self.config.source_URL)
#             with open(self.config.local_data_file, "wb") as filename:
#                 filename.write(data.content)
#             logger.info(f"{filename} download! with following info:") #\n{headers}
#         else:
#             logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

#     def extract_zip_file(self):
#         """zip file_path:str
#         Extracts the zip file into the data directory
#         Function returns None
#         """
#         unzip_path = self.config.unzip_dir
#         os.makedirs(unzip_path, exist_ok=True)
#         with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
#             zip_ref.extractall(unzip_path)

In [78]:
class DataIngestion:
	def __init__(self, config: DataIngestionConfig):
		self.config = config

	def download_file(self):
		if os.path.exists(self.config.local_data_file):
			if zipfile.is_zipfile(self.config.local_data_file) and os.path.getsize(self.config.local_data_file) > 0:
				logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")
				return
			os.remove(self.config.local_data_file)

		resp = requests.get(self.config.source_URL, stream=True)
		resp.raise_for_status()

		with open(self.config.local_data_file, "wb") as f:
			for chunk in resp.iter_content(chunk_size=8192):
				if chunk:
					f.write(chunk)
		logger.info(f"Downloaded file: {self.config.local_data_file} with headers: {resp.headers}")


		if not zipfile.is_zipfile(self.config.local_data_file):
			raise RuntimeError("Downloaded file is not a valid zip. Check the source URL/content.")

	def extract_zip_file(self):
		if not zipfile.is_zipfile(self.config.local_data_file):
			raise RuntimeError("Cannot extract: downloaded file is not a valid zip.")
		unzip_path = self.config.unzip_dir
		os.makedirs(unzip_path, exist_ok=True)
		with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
			zip_ref.extractall(unzip_path)

In [79]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-19 18:23:55,994: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-03-19 18:23:55,997: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-19 18:23:56,000: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-19 18:23:56,001: INFO: common: created directory at: artifacts]
[2026-03-19 18:23:56,002: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-19 18:23:56,005: INFO: 1124021287: File already exists of size: ~24 KB]
